# NLP Lab Task 5 — N-gram Language Models
**Instructions:** Run the cells from top to bottom. Uses only Python's standard library. Probabilities and log values are rounded to 4 decimal places for display. The vocabulary is built from all four training sentences and includes `</s>` but excludes `<s>`. Sentence probabilities include the end marker.

In [1]:
# Task 1: Tokenization
from collections import Counter
from itertools import permutations
from math import log, exp, inf
from functools import reduce
import math

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]
test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

train_tok = [tokenize(s) for s in train]
test_tok = [tokenize(s) for s in test]

print("Tokenized training sentences:")
for tokens in train_tok:
    print(tokens)
print("\\nTokenized test sentences:")
for tokens in test_tok:
    print(tokens)

with_markers = sum(len(tokens) for tokens in train_tok)
without_markers = sum(len(s.lower().split(" ")) for s in train)
print("\\nTraining tokens with markers:", with_markers)
print("Training tokens without markers:", without_markers)

def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        # Accept either raw sentences or already-tokenized lists.
        words = sentence.lower().split(" ") if isinstance(sentence, str) else sentence
        vocab.update(words)
    vocab.discard("<s>")
    vocab.add("</s>")
    return vocab

V_set = build_vocab(train)
V = sorted(V_set)
print("\\nVocabulary V:", V_set)
print("Sorted vocabulary:", V)
print("Vocabulary size:", len(V))

Tokenized training sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
\nTokenized test sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']
\nTraining tokens with markers: 30
Training tokens without markers: 22
\nVocabulary V: {'the', 'cat', '</s>', 'mat', 'rug', 'ate', 'on', 'bone', 'sat', 'fish', 'dog'}
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']
Vocabulary size: 11


In [2]:
# Task 2: Generating N-grams
def get_ngrams(tokens, n):
    if n <= 0:
        raise ValueError("n must be a positive integer")
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

first_sentence = train_tok[0]
for n in (1, 2, 3):
    print(f"{n}-grams of the first training sentence:")
    print(get_ngrams(first_sentence, n))

print("\\nVerify m - n + 1 counts for every training sentence:")
for index, tokens in enumerate(train_tok, start=1):
    m = len(tokens)
    for n in (1, 2, 3):
        actual = len(get_ngrams(tokens, n))
        expected = max(0, m - n + 1)
        print(f"Sentence {index}: m={m}, n={n}, actual={actual}, expected={expected}")
        assert actual == expected

short_tokens = ["one", "two"]
print("\\nWhen n is longer than the sentence:")
print("Tokens:", short_tokens)
print("get_ngrams(tokens, 3) =", get_ngrams(short_tokens, 3))

1-grams of the first training sentence:
[('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
2-grams of the first training sentence:
[('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
3-grams of the first training sentence:
[('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]
\nVerify m - n + 1 counts for every training sentence:
Sentence 1: m=8, n=1, actual=8, expected=8
Sentence 1: m=8, n=2, actual=7, expected=7
Sentence 1: m=8, n=3, actual=6, expected=6
Sentence 2: m=7, n=1, actual=7, expected=7
Sentence 2: m=7, n=2, actual=6, expected=6
Sentence 2: m=7, n=3, actual=5, expected=5
Sentence 3: m=8, n=1, actual=8, expected=8
Sentence 3: m=8, n=2, actual=7, expected=7
Sentence 3: m=8, n=3, actual=6, expected=6
Sentence 4: m=7, n=1, actual=7, expected=7
Sentence 4: m=7, n=2, actual=6, expected=6
Sentence 4: m=7, n=3, 

In [3]:
# Task 3: Counting N-grams
all_train_tokens = [token for sentence in train_tok for token in sentence]
unigram_counts = Counter(all_train_tokens)
bigram_counts = Counter(ngram for sentence in train_tok for ngram in get_ngrams(sentence, 2))
trigram_counts = Counter(ngram for sentence in train_tok for ngram in get_ngrams(sentence, 3))

def print_counts(title, counts):
    print(title)
    for ngram, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
        print(f"{str(ngram):30s} {count}")

print_counts("Unigram counts (highest to lowest):", unigram_counts)
print()
print_counts("Bigram counts (highest to lowest):", bigram_counts)
print()
print_counts("Trigram counts (highest to lowest):", trigram_counts)

print("\\nFive most common bigrams:")
for ngram, count in bigram_counts.most_common(5):
    print(ngram, count)

print("\\nDistinct n-gram counts:")
print("Unigrams:", len(unigram_counts))
print("Bigrams:", len(bigram_counts))
print("Trigrams:", len(trigram_counts))

def count_of(ngram):
    # ngram is a tuple, e.g. ("the", "cat")
    if len(ngram) == 1:
        return unigram_counts.get(ngram[0], 0)
    if len(ngram) == 2:
        return bigram_counts.get(tuple(ngram), 0)
    if len(ngram) == 3:
        return trigram_counts.get(tuple(ngram), 0)
    return 0

print("\\nSeen bigram ('the', 'cat'):", count_of(("the", "cat")))
print("Unseen bigram ('fish', 'bone'):", count_of(("fish", "bone")))

Unigram counts (highest to lowest):
the                            8
</s>                           4
<s>                            4
ate                            2
cat                            2
dog                            2
on                             2
sat                            2
bone                           1
fish                           1
mat                            1
rug                            1

Bigram counts (highest to lowest):
('<s>', 'the')                 4
('ate', 'the')                 2
('on', 'the')                  2
('sat', 'on')                  2
('the', 'cat')                 2
('the', 'dog')                 2
('bone', '</s>')               1
('cat', 'ate')                 1
('cat', 'sat')                 1
('dog', 'ate')                 1
('dog', 'sat')                 1
('fish', '</s>')               1
('mat', '</s>')                1
('rug', '</s>')                1
('the', 'bone')                1
('the', 'fish')                1
('th

In [4]:
# Task 4: Unigram Model (MLE)
N_unigrams = sum(unigram_counts.values())

def unigram_prob(w):
    return unigram_counts.get((w,), 0) / N_unigrams

# unigram_counts uses strings as keys, so use this implementation:
def unigram_prob(w):
    return unigram_counts.get(w, 0) / N_unigrams

print("Unigram probabilities (MLE):")
print(f"{'Word':12s} {'Count':>5s} {'Probability':>12s}")
for word in sorted(unigram_counts):
    print(f"{word:12s} {unigram_counts[word]:5d} {unigram_prob(word):12.4f}")
assert abs(sum(unigram_prob(w) for w in unigram_counts) - 1.0) < 1e-12
print("Sum of unigram probabilities = 1.0000")

def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    # Exclude <s>; include </s> as an ordinary modeled token.
    return math.prod(unigram_prob(w) for w in tokens[1:])

print("\\nUnigram sentence probabilities:")
for sentence in test:
    print(repr(sentence), f"{unigram_sentence_prob(sentence):.4f}")

words = test[0].lower().split()
permutation_probs = {}
for order in permutations(words):
    p = math.prod(unigram_prob(w) for w in order) * unigram_prob("</s>")
    permutation_probs.setdefault(round(p, 12), []).append(" ".join(order))
print("\\nPermutation demonstration for the first test sentence:")
for p, orders in permutation_probs.items():
    print(f"Probability={p:.8f}; number of orders with this probability={len(orders)}")
    print("Examples:", orders[:3])
print("Because unigram probability depends on word counts, reordering the same words gives the same probability.")

Unigram probabilities (MLE):
Word         Count  Probability
</s>             4       0.1333
<s>              4       0.1333
ate              2       0.0667
bone             1       0.0333
cat              2       0.0667
dog              2       0.0667
fish             1       0.0333
mat              1       0.0333
on               2       0.0667
rug              1       0.0333
sat              2       0.0667
the              8       0.2667
Sum of unigram probabilities = 1.0000
\nUnigram sentence probabilities:
'the cat sat on the rug' 0.0000
'the fish sat on the mat' 0.0000
\nPermutation demonstration for the first test sentence:
Probability=0.00000009; number of orders with this probability=720
Examples: ['the cat sat on the rug', 'the cat sat on rug the', 'the cat sat the on rug']
Because unigram probability depends on word counts, reordering the same words gives the same probability.


In [5]:
# Task 5: Bigram Model (MLE)
bigram_context_counts = Counter()
for (prev, word), count in bigram_counts.items():
    bigram_context_counts[prev] += count

def bigram_prob(prev, w):
    denominator = bigram_context_counts.get(prev, 0)
    if denominator == 0:
        return 0.0
    return bigram_counts.get((prev, w), 0) / denominator

def next_word_distribution(prev):
    denominator = bigram_context_counts.get(prev, 0)
    if denominator == 0:
        return {}
    return {
        word: bigram_counts.get((prev, word), 0) / denominator
        for word in sorted(V_set)
        if bigram_counts.get((prev, word), 0) > 0
    }

dist_the = next_word_distribution("the")
print("Next-word distribution for 'the':")
for word, probability in dist_the.items():
    print(f"P({word} | the) = {probability:.4f}")
assert abs(sum(dist_the.values()) - 1.0) < 1e-12

def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    return math.prod(bigram_prob(tokens[i-1], tokens[i]) for i in range(1, len(tokens)))

print("\\nBigram sentence probabilities (MLE):")
for sentence in test:
    print(repr(sentence), f"{bigram_sentence_prob(sentence):.4f}")

def predict_next(prev):
    distribution = next_word_distribution(prev)
    if not distribution:
        return None
    best_probability = max(distribution.values())
    return min(word for word, p in distribution.items() if p == best_probability)

print("\\nNext-word predictions:")
for prev in ["<s>", "the", "cat", "dog", "sat", "fish"]:
    print(f"predict_next({prev!r}) = {predict_next(prev)!r}")

Next-word distribution for 'the':
P(bone | the) = 0.1250
P(cat | the) = 0.2500
P(dog | the) = 0.2500
P(fish | the) = 0.1250
P(mat | the) = 0.1250
P(rug | the) = 0.1250
\nBigram sentence probabilities (MLE):
'the cat sat on the rug' 0.0156
'the fish sat on the mat' 0.0000
\nNext-word predictions:
predict_next('<s>') = 'the'
predict_next('the') = 'cat'
predict_next('cat') = 'ate'
predict_next('dog') = 'ate'
predict_next('sat') = 'on'
predict_next('fish') = '</s>'


In [6]:
# Task 6: Trigram Model
trigram_context_counts = Counter()
for (w1, w2, w3), count in trigram_counts.items():
    trigram_context_counts[(w1, w2)] += count

def trigram_prob(w1, w2, w3):
    denominator = bigram_counts.get((w1, w2), 0)
    if denominator == 0:
        return 0.0
    return trigram_counts.get((w1, w2, w3), 0) / denominator

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    if len(tokens) < 2:
        return 1.0
    # First predicted token uses bigram model P(w1 | <s>).
    probability = bigram_prob(tokens[0], tokens[1])
    # All subsequent tokens, including </s>, use the trigram model.
    for i in range(2, len(tokens)):
        probability *= trigram_prob(tokens[i-2], tokens[i-1], tokens[i])
    return probability

print("Trigram sentence probabilities:")
for sentence in test:
    print(repr(sentence), f"{trigram_sentence_prob(sentence):.4f}")

print("\\nDistinct n-grams:")
print(f"{'N-gram type':15s} {'Distinct count':>15s}")
print(f"{'Unigrams':15s} {len(unigram_counts):15d}")
print(f"{'Bigrams':15s} {len(bigram_counts):15d}")
print(f"{'Trigrams':15s} {len(trigram_counts):15d}")

def unseen_ngrams(tokens, n, counts):
    return [ng for ng in get_ngrams(tokens, n) if counts.get(ng, 0) == 0]

print("\\nUnseen bigrams/trigrams per test sentence:")
for sentence in test:
    tokens = tokenize(sentence)
    unseen_bi = unseen_ngrams(tokens, 2, bigram_counts)
    unseen_tri = unseen_ngrams(tokens, 3, trigram_counts)
    print(f"Sentence: {sentence}")
    print("  Unseen bigrams:", unseen_bi)
    print("  Unseen trigrams:", unseen_tri)
    if len(unseen_bi) > len(unseen_tri):
        comparison = "bigrams"
    elif len(unseen_tri) > len(unseen_bi):
        comparison = "trigrams"
    else:
        comparison = "equal number"
    print("  More unseen N-grams:", comparison)

Trigram sentence probabilities:
'the cat sat on the rug' 0.1250
'the fish sat on the mat' 0.0000
\nDistinct n-grams:
N-gram type      Distinct count
Unigrams                     12
Bigrams                      18
Trigrams                     19
\nUnseen bigrams/trigrams per test sentence:
Sentence: the cat sat on the rug
  Unseen bigrams: []
  Unseen trigrams: []
  More unseen N-grams: equal number
Sentence: the fish sat on the mat
  Unseen bigrams: [('fish', 'sat')]
  Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
  More unseen N-grams: trigrams


In [7]:
# Task 7: Zero-Probability Problem
def unseen_bigrams(sentence):
    tokens = tokenize(sentence)
    return [ng for ng in get_ngrams(tokens, 2) if bigram_counts.get(ng, 0) == 0]

print("Unseen bigrams and bigram MLE sentence probabilities:")
for sentence in test:
    print(f"Sentence: {sentence}")
    print("  Unseen bigrams:", unseen_bigrams(sentence))
    print(f"  Bigram MLE probability: {bigram_sentence_prob(sentence):.4f}")

print("\\nCount unseen possible word pairs (markers excluded):")
word_types = sorted(set(word for s in train for word in s.lower().split()))
# All ordered pairs from the observed word types; markers are excluded.
all_possible_pairs = [(a, b) for a in word_types for b in word_types]
observed_word_pairs = {
    (a, b)
    for sentence in train
    for a, b in zip(sentence.lower().split(), sentence.lower().split()[1:])
}
unseen_pair_count = sum(pair not in observed_word_pairs for pair in all_possible_pairs)
total_pair_count = len(all_possible_pairs)
print("Word types:", len(word_types))
print("Possible ordered pairs:", total_pair_count)
print("Unseen ordered pairs:", unseen_pair_count)
print(f"Unseen pair percentage: {100 * unseen_pair_count / total_pair_count:.2f}%")

def safe_log(p):
    return -math.inf if p == 0 else math.log(p)

second_test_probability = bigram_sentence_prob(test[1])
print("\\nLog probability of second test sentence (bigram MLE):", safe_log(second_test_probability))

Unseen bigrams and bigram MLE sentence probabilities:
Sentence: the cat sat on the rug
  Unseen bigrams: []
  Bigram MLE probability: 0.0156
Sentence: the fish sat on the mat
  Unseen bigrams: [('fish', 'sat')]
  Bigram MLE probability: 0.0000
\nCount unseen possible word pairs (markers excluded):
Word types: 10
Possible ordered pairs: 100
Unseen ordered pairs: 87
Unseen pair percentage: 87.00%
\nLog probability of second test sentence (bigram MLE): -inf


In [8]:
# Task 8: Smoothing
V_size = len(V_set)

def bigram_prob_addone(prev, w):
    return (bigram_counts.get((prev, w), 0) + 1) / (bigram_context_counts.get(prev, 0) + V_size)

print("MLE vs add-one smoothing:")
for prev, word in [("fish", "sat"), ("the", "cat")]:
    print(f"P({word} | {prev}): MLE={bigram_prob(prev, word):.4f}, add-one={bigram_prob_addone(prev, word):.4f}")

# Check every possible previous word in V; the distribution over V sums to one.
for prev in V:
    smoothed_total = sum(bigram_prob_addone(prev, word) for word in V)
    assert abs(smoothed_total - 1.0) < 1e-12, (prev, smoothed_total)
print("Add-one distributions sum to 1 for every previous word in V.")

def bigram_prob_addk(prev, w, k):
    if k <= 0:
        raise ValueError("k must be greater than 0")
    return (bigram_counts.get((prev, w), 0) + k) / (bigram_context_counts.get(prev, 0) + k * V_size)

print("\\nP(sat | fish) under add-k smoothing:")
print(f"{'k':>8s} {'Probability':>12s}")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:8.2f} {bigram_prob_addk('fish', 'sat', k):12.4f}")

MLE vs add-one smoothing:
P(sat | fish): MLE=0.0000, add-one=0.0833
P(cat | the): MLE=0.2500, add-one=0.1579
Add-one distributions sum to 1 for every previous word in V.
\nP(sat | fish) under add-k smoothing:
       k  Probability
    1.00       0.0833
    0.50       0.0769
    0.10       0.0476
    0.01       0.0090


In [9]:
# Task 9: Perplexity and Model Comparison
# Perplexity is calculated over predicted tokens only (including </s>, excluding <s>).
# A zero probability makes perplexity infinite.

def unigram_addone_prob(w):
    return (unigram_counts.get(w, 0) + 1) / (N_unigrams + V_size)

def sentence_prob_from_fn(sentence, prob_fn, order):
    tokens = tokenize(sentence)
    if order == "bigram":
        return math.prod(prob_fn(tokens[i-1], tokens[i]) for i in range(1, len(tokens)))
    if order == "unigram":
        return math.prod(prob_fn(w) for w in tokens[1:])
    raise ValueError("order must be 'unigram' or 'bigram'")

def perplexity(sentences, prob_fn, order="bigram"):
    log_probability_sum = 0.0
    token_count = 0
    for sentence in sentences:
        tokens = tokenize(sentence)
        predicted = tokens[1:]  # words + </s>; do not score <s>
        token_count += len(predicted)
        if order == "bigram":
            probabilities = [prob_fn(tokens[i-1], tokens[i]) for i in range(1, len(tokens))]
        elif order == "unigram":
            probabilities = [prob_fn(w) for w in predicted]
        else:
            raise ValueError("order must be 'unigram' or 'bigram'")
        if any(p == 0 for p in probabilities):
            return math.inf
        log_probability_sum += sum(math.log(p) for p in probabilities)
    return math.exp(-log_probability_sum / token_count) if token_count else math.inf

models = [
    ("Bigram MLE", lambda a, b: bigram_prob(a, b), "bigram"),
    ("Bigram add-one", lambda a, b: bigram_prob_addone(a, b), "bigram"),
    ("Unigram add-one", lambda w: unigram_addone_prob(w), "unigram")
]
print("Perplexity on both test sentences:")
print(f"{'Model':20s} {'Perplexity':>12s}")
for name, fn, order in models:
    value = perplexity(test, fn, order)
    print(f"{name:20s} {value:12.4f}" if math.isfinite(value) else f"{name:20s} {'infinity':>12s}")

print("\\nAdd-k bigram perplexity on both test sentences:")
k_results = []
for k in [1, 0.5, 0.1, 0.01]:
    value = perplexity(test, lambda a, b, k=k: bigram_prob_addk(a, b, k), "bigram")
    k_results.append((k, value))
    print(f"k={k:<5} perplexity={value:.4f}" if math.isfinite(value) else f"k={k:<5} perplexity=infinity")
best_test_k, best_test_pp = min(k_results, key=lambda item: item[1])
print(f"Best k directly on test set: {best_test_k} (perplexity={best_test_pp:.4f})")

# Validation experiment: train on first three sentences only, retain V from all four.
validation_sentence = train[3]
validation_train_tok = train_tok[:3]
val_unigram_counts = Counter(w for sentence in validation_train_tok for w in sentence)
val_bigram_counts = Counter(ng for sentence in validation_train_tok for ng in get_ngrams(sentence, 2))
val_bigram_context_counts = Counter()
for (prev, word), count in val_bigram_counts.items():
    val_bigram_context_counts[prev] += count
val_N = sum(val_unigram_counts.values())

def val_bigram_prob_addk(prev, w, k):
    return (val_bigram_counts.get((prev, w), 0) + k) / (val_bigram_context_counts.get(prev, 0) + k * V_size)

def val_perplexity(sentence, k):
    tokens = tokenize(sentence)
    probs = [val_bigram_prob_addk(tokens[i-1], tokens[i], k) for i in range(1, len(tokens))]
    return math.exp(-sum(math.log(p) for p in probs) / len(probs)) if all(p > 0 for p in probs) else math.inf

validation_results = [(k, val_perplexity(validation_sentence, k)) for k in [1, 0.5, 0.1, 0.01]]
print("\\nValidation results (train first 3 sentences, validate on 4th):")
for k, value in validation_results:
    print(f"k={k:<5} perplexity={value:.4f}" if math.isfinite(value) else f"k={k:<5} perplexity=infinity")
best_validation_k, best_validation_pp = min(validation_results, key=lambda item: item[1])
test_pp_with_validation_k = perplexity(test, lambda a, b: bigram_prob_addk(a, b, best_validation_k), "bigram")
print(f"Best k selected by validation: {best_validation_k}")
print(f"Test perplexity using validation-selected k: {test_pp_with_validation_k:.4f}")
print(f"Best test-set-selected k: {best_test_k}")
print(f"Test perplexity using test-set-selected k: {best_test_pp:.4f}")
print("Note: selecting k directly on the test set is optimistic; validation-based selection is the proper evaluation practice.")

Perplexity on both test sentences:
Model                  Perplexity
Bigram MLE               infinity
Bigram add-one             5.7969
Unigram add-one           10.1247
\nAdd-k bigram perplexity on both test sentences:
k=1     perplexity=5.7969
k=0.5   perplexity=4.5703
k=0.1   perplexity=2.9559
k=0.01  perplexity=2.6272
Best k directly on test set: 0.01 (perplexity=2.6272)
\nValidation results (train first 3 sentences, validate on 4th):
k=1     perplexity=8.5857
k=0.5   perplexity=8.0163
k=0.1   perplexity=8.0260
k=0.01  perplexity=13.1304
Best k selected by validation: 0.5
Test perplexity using validation-selected k: 4.5703
Best test-set-selected k: 0.01
Test perplexity using test-set-selected k: 2.6272
Note: selecting k directly on the test set is optimistic; validation-based selection is the proper evaluation practice.
